# Float64 rectum segmentation with predicted star-kernel vantage points
Two-stage pipeline: MRI → frozen pretrained kernel U-Net → largest-component distance-transform centre → trainable rectum U-Net → existing DeepFlow star LBP layer → rectum probabilities. The rectum mask is supervision only. The LBP layer uses directional hard-star constraints as implemented in the DeepFlow PyTorch LBP solver, with differentiable soft marginals; thresholded marginals are not the same as hard MAP. The supplied layer is axis-wise, not an exact Euclidean Bresenham visibility constraint.

The original kernel validation patients are reused for development validation, not claimed as an independent test set. For an unbiased test, use genuinely external patients; for leakage-free training centres use out-of-fold kernel checkpoints (not supplied). No kernel training is performed here.

This run trains the rectum U-Net and soft star layer **from scratch in float64**.
The frozen pretrained kernel predictor and preprocessing/cache tensors remain float32;
training batches are converted at consumption. No rectum-model checkpoint is loaded.
A separate output directory keeps this experiment distinct from the float32 run.


## 1. Configuration and environment
Set `MASK_DIR` to the actual binary rectum annotation directory. The original preprocessing worker is required for exact normalization; place it beside the notebook or set its path. DeepFlow must be built in the active Python environment. The checkpoint is the original rectum kernel checkpoint, not the synthetic segmentation checkpoint. Default: evaluate only rectum-positive slices (annotation-defined cohort, not a full-volume detection benchmark); set `INCLUDE_EMPTY_RECTUM=True` for all slices. Empty predicted kernels use image-only argmax fallback and are explicitly flagged.

In [ ]:
from pathlib import Path
IMAGE_DIR = Path('../Datasets/MOGaMBOv0/images')
MASK_DIR = Path('../Datasets/MOGaMBOv0/masks/rectum')  # EDIT: binary rectum NIfTI directory
KERNEL_DIR = Path('../Datasets/MOGaMBOv0/kernels/rectum_2d_margin_5')
KERNEL_CHECKPOINT = Path('./runs/kernel_segmentation_larger_deeper/rectum/best_star_kernel_unet_2d.pt')
OUTPUT_DIR = Path('./runs/rectum_predicted_center_star_rot90_float64_from_scratch')
WORKER_MODULE_DIR = Path.cwd()
SLICE_AXIS = 2
INCLUDE_EMPTY_RECTUM = False
KERNEL_BATCH_SIZE = 32
ROTATE_K = -1  # displayed left -> top with imshow(origin="upper")
CACHE_VERSION = 2
GPU_DATASET = True
USE_AMP_UNET = False  # Float64 training does not use AMP.
CHECK_EVERY = 50
CENTER_COMPONENT_CONNECTIVITY = 8
IMAGE_SUFFIXES = ("_image", "_img", "_mri", "_t2", "_t1", "_T2", "_T2FS", "_pat")
KERNEL_SUFFIXES = ("_kernel", "_kernels", "_mask", "_label", "_labels", "_seg", "_uterus", "_ut", "_pat")
MASK_SUFFIXES = ("_rectum_mask", "_rectum", "_mask", "_label", "_labels", "_seg", "_pat")
VAL_FRACTION = 0.20
RECTUM_LABEL = None  # None: require binary 0/1 or 0/255; otherwise explicit label ID
from pathlib import Path
import copy
import random
import sys
import numpy as np
import torch
TRAIN_DTYPE = torch.float64
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
 # from binary_deepflow import BinaryDeepFlow2dStar

DEEPFLOW_PYTORCH = None #"../Deepflow/pytorch"
if DEEPFLOW_PYTORCH is None:
    candidates = []
    for root in (Path.cwd(), *Path.cwd().parents):
        candidates.extend((root / "DeepFlow" / "pytorch", root / "pytorch", root))
    DEEPFLOW_PYTORCH = next(
        (p for p in candidates if (p / "binary_deepflow.py").is_file()), None
    )
if DEEPFLOW_PYTORCH is None:
    raise FileNotFoundError("Set DEEPFLOW_PYTORCH to the repository's DeepFlow/pytorch directory.")
sys.path.insert(0, str(Path(DEEPFLOW_PYTORCH).resolve()))
try:
    import deepflow
    from binary_deepflow import BinaryDeepFlow2dStar
except ImportError as exc:
    raise ImportError(
        "Build/install DeepFlow in this kernel's environment; see the setup cell above."
    ) from exc

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE = 128
BATCH_SIZE = 4
EPOCHS = 1  # Inspect the initial float64 epoch before extending the run.
LEARNING_RATE = 1e-3
BASE_CHANNELS = 24
PAIRWISE_WEIGHT = 0.15
# DeepFlow LBP controls; independent of notebook CHECK_EVERY validation checks.
STAR_CHECK_EVERY = 100
STAR_COMPILE_UPDATES = True  # Compile tensor update blocks; first call includes compilation.
STAR_RECORD_HISTORY = False  # Normal training retains final summaries only.
SCORE_SCALE = 4.0  # Bound unary scores for numerical stability; positive favours foreground.
CHECKPOINT_PATH = OUTPUT_DIR / "unet_star_best.pt"
print("Python:", sys.executable)
print("Device:", DEVICE, "| Training dtype:", TRAIN_DTYPE, "| DeepFlow:", deepflow.__file__)
import nibabel as nib
from nibabel.processing import resample_from_to
from scipy import ndimage as ndi
from tqdm.auto import tqdm
import json
sys.path.insert(0, str(WORKER_MODULE_DIR.resolve()))
import star_kernel_segmentation_preprocess_worker as prep
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
import os, hashlib, time
from concurrent.futures import ThreadPoolExecutor
CPU_WORKERS = min(4, os.cpu_count() or 1)
torch.set_num_threads(1)  # avoid nested CPU thread oversubscription
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


## 2. Frozen kernel predictor — original architecture and checkpoint

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class ConvBlockGN(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_ch),
            nn.LeakyReLU(0.1, inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_ch),
            nn.LeakyReLU(0.1, inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class SmallUNetSegmenter(nn.Module):
    def __init__(self, base_channels=16):
        super().__init__()
        b = base_channels
        self.enc1 = ConvBlockGN(1, b)
        self.enc2 = ConvBlockGN(b, 2*b)
        self.enc3 = ConvBlockGN(2*b, 4*b)
        self.bottleneck = ConvBlockGN(4*b, 8*b)
        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(8*b, 4*b, 2, stride=2)
        self.dec3 = ConvBlockGN(8*b, 4*b)
        self.up2 = nn.ConvTranspose2d(4*b, 2*b, 2, stride=2)
        self.dec2 = ConvBlockGN(4*b, 2*b)
        self.up1 = nn.ConvTranspose2d(2*b, b, 2, stride=2)
        self.dec1 = ConvBlockGN(2*b, b)
        self.head = nn.Conv2d(b, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        z = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(z), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)  # logits, NOT probabilities

checkpoint = torch.load(KERNEL_CHECKPOINT, map_location='cpu', weights_only=True)
SIZE = int(checkpoint['size'])
IMAGE_SIZE = SIZE
PRED_THRESHOLD = float(checkpoint['pred_threshold'])
kernel_model = SmallUNetSegmenter(int(checkpoint['base_channels'])).to(DEVICE)
kernel_model.load_state_dict(checkpoint['model_state_dict'], strict=True)
kernel_model.eval().requires_grad_(False)
print('Frozen kernel checkpoint:', KERNEL_CHECKPOINT, 'epoch:', checkpoint['epoch'], 'size:', SIZE)


In [ ]:
def resize_2d(array,is_mask=False):
    tensor=torch.from_numpy(np.asarray(array,dtype=np.float32).copy())[None,None]
    if is_mask:return (F.interpolate(tensor,size=(SIZE,SIZE),mode='nearest')[0,0].numpy()>0.5)
    return F.interpolate(tensor,size=(SIZE,SIZE),mode='bilinear',align_corners=False)[0,0].numpy()

from scipy import ndimage as ndi

def center_from_predicted_kernel(predicted_mask):
    """Return (x,y) pixel center from the largest 8-connected component, or None.

    Tie-breaking is deterministic: np.argmax chooses first max in row-major order.
    """
    binary = np.asarray(predicted_mask, dtype=bool)
    if not binary.any():
        return None
    structure = ndi.generate_binary_structure(2, 2 if CENTER_COMPONENT_CONNECTIVITY == 8 else 1)
    components, n_components = ndi.label(binary, structure=structure)
    sizes = np.bincount(components.ravel())
    sizes[0] = 0
    largest = components == int(np.argmax(sizes))
    interior_distance = ndi.distance_transform_edt(largest)
    y, x = np.unravel_index(int(np.argmax(interior_distance)), binary.shape)
    return np.array([x, y], dtype=np.float32)


@torch.inference_mode()
def predict_centres(images):
    # images are already resized and normalized exactly once.
    probabilities = kernel_model(images.to(DEVICE, dtype=torch.float32, non_blocking=True)).sigmoid().cpu().numpy()[:,0]
    def extract(p):
        if not np.isfinite(p).all(): raise FloatingPointError('Non-finite kernel prediction')
        xy = center_from_predicted_kernel(p >= PRED_THRESHOLD)
        fallback = xy is None
        if xy is None:
            row, col = np.unravel_index(p.argmax(), p.shape)
            centre = [row, col]
        else:
            centre = xy[::-1].copy()  # (x,y) -> (row,column)
        return centre, fallback
    with ThreadPoolExecutor(max_workers=CPU_WORKERS) as pool:
        centres, fallback = zip(*pool.map(extract, probabilities))
    return probabilities, np.asarray(centres, dtype=np.float32), np.asarray(fallback)


## 3. Pair volumes and reproduce the original patient split
Kernel files are used to reconstruct the original patient cohort and to audit centres, never to select a predicted centre. Check that the original training run used the same seed, fraction and patient list; its checkpoint does not store those IDs.

In [ ]:
def patient_id(path: Path, suffixes):
    name = path.name
    if not name.lower().endswith(".nii.gz"):
        raise ValueError(f"Expected .nii.gz: {path}")
    stem = name[:-7].lower()
    for suffix in sorted(suffixes, key=len, reverse=True):
        if stem.endswith(suffix.lower()):
            return stem[:-len(suffix)]
    return stem


def find_volumes(root, suffixes):
    if not root.is_dir():
        raise FileNotFoundError(f"Directory does not exist: {root}")
    files = sorted(p for p in root.rglob("*") if p.is_file() and p.name.lower().endswith(".nii.gz"))
    if not files:
        raise FileNotFoundError(f"No .nii.gz volumes found in {root}")
    result = {}
    for p in files:
        pid = patient_id(p, suffixes)
        if pid in result:
            raise ValueError(f"Duplicate patient ID '{pid}': {result[pid]} and {p}; refine suffix pairing")
        result[pid] = p
    return result



image_files = find_volumes(IMAGE_DIR, IMAGE_SUFFIXES)
kernel_files = find_volumes(KERNEL_DIR, KERNEL_SUFFIXES)
mask_files = find_volumes(MASK_DIR, MASK_SUFFIXES)
if not (set(image_files) == set(kernel_files) == set(mask_files)):
    raise ValueError(f'Patient mismatch: missing masks {set(image_files)-set(mask_files)}, missing kernels {set(image_files)-set(kernel_files)}, extra masks {set(mask_files)-set(image_files)}, extra kernels {set(kernel_files)-set(image_files)}')
patient_ids = sorted(image_files)
assert len(patient_ids) >= 2
rng=np.random.default_rng(SEED)
shuffled=rng.permutation(patient_ids)
n_val=max(1,min(len(shuffled)-1,int(round(len(shuffled)*VAL_FRACTION))))
val_patients=set(shuffled[:n_val].tolist());train_patients=set(shuffled[n_val:].tolist())
assert train_patients.isdisjoint(val_patients)


print('Train patients:', sorted(train_patients))
print('Validation patients:', sorted(val_patients))
(OUTPUT_DIR/'splits.json').write_text(json.dumps({'train':sorted(train_patients),'val':sorted(val_patients)},indent=2))


## 4. Preprocess and predict centres once
Canonical RAS MRI is the reference grid. Resample annotations to MRI geometry with nearest-neighbour interpolation, then resize slices to the kernel network resolution. MRI normalization is delegated to the original worker. The full predicted kernel probability map is kept for QC; the second model consumes only MRI and predicted centre. Ground-truth kernel is used solely for validity diagnostics. The arrays are held in RAM for this run.
All MRI, rectum and kernel slices are rotated 90° clockwise in the displayed array (left becomes top) before resize. Patient preprocessing and exact SciPy centre extraction use bounded CPU thread pools; GPU inference is batched. Persistent caches are invalidated by input/settings changes.

In [ ]:
def aligned_label(path, ref):
    label = nib.as_closest_canonical(nib.load(str(path)))
    if label.shape != ref.shape or not np.allclose(label.affine, ref.affine):
        label = resample_from_to(label, (ref.shape, ref.affine), order=0)
    return np.asarray(label.dataobj)

def load_patient(pid):
    ref = nib.as_closest_canonical(nib.load(str(image_files[pid])))
    if len(ref.shape) != 3: raise ValueError(f'{pid}: expected 3D MRI')
    image = np.asarray(ref.dataobj, dtype=np.float32)
    mask = aligned_label(mask_files[pid], ref)
    kernel = aligned_label(kernel_files[pid], ref) > 0
    if RECTUM_LABEL is None:
        if not np.isin(np.unique(mask), [0,1,255]).all():
            raise ValueError(f'{pid}: nonbinary mask; set RECTUM_LABEL')
        mask = mask > 0
    else: mask = mask == RECTUM_LABEL
    # Rotate all three arrays together BEFORE resizing/normalization.
    def planes(v):
        return np.ascontiguousarray(np.rot90(np.moveaxis(v, SLICE_AXIS, 0), ROTATE_K, axes=(1,2)))
    image, mask, kernel = map(planes, (image, mask, kernel))
    def resize_stack(v, mode):
        t = torch.from_numpy(v.astype(np.float32, copy=False))[:,None]
        kw = {'align_corners':False} if mode == 'bilinear' else {}
        return F.interpolate(t, size=(SIZE,SIZE), mode=mode, **kw)[:,0].numpy()
    ys = resize_stack(mask, 'nearest') > .5
    zs = np.flatnonzero(ys.any(axis=(1,2)) | INCLUDE_EMPTY_RECTUM)
    if not len(zs): return None
    xs = resize_stack(image[zs], 'bilinear')
    xs = np.asarray([prep.normalize_slice(v) for v in xs], dtype=np.float32)
    if not np.isfinite(xs).all(): raise ValueError(f'{pid}: invalid MRI')
    x = torch.from_numpy(xs[:,None])
    y = torch.from_numpy(ys[zs,None].astype(np.float32))
    k = resize_stack(kernel[zs], 'nearest') > .5
    return x,y,k,zs.tolist()

def assemble_patient(pid, data):
    if data is None: return None
    x,y,k,zs = data
    predictions = [predict_centres(batch) for batch in x.split(KERNEL_BATCH_SIZE)]
    p,c,f = (np.concatenate(items) for items in zip(*predictions))
    assert np.isfinite(c).all() and ((c >= 0) & (c < SIZE)).all()
    records=[]
    for i,z in enumerate(zs):
        r,col = c[i].astype(int)
        records.append({'patient':pid,'slice':z,'fallback':bool(f[i]),
                        'gt_kernel_defined':bool(k[i].any()),'centre_in_kernel':bool(k[i,r,col]),
                        'centre_in_rectum':bool(y[i,0,r,col]),'rectum_present':bool(y[i].any())})
    return x,y,torch.from_numpy(c),p.astype(np.float16),records

class RectumDataset(Dataset):
    def __init__(self, patients):
        data = []
        # Cache key covers source files, checkpoint, orientation and preprocessing settings.
        def fingerprint(pid):
            paths = [image_files[pid], mask_files[pid], kernel_files[pid], KERNEL_CHECKPOINT, Path(prep.__file__)]
            metadata = [(str(p.resolve()),p.stat().st_size,p.stat().st_mtime_ns) for p in paths]
            return hashlib.sha256(repr((metadata,CACHE_VERSION,ROTATE_K,SIZE,SLICE_AXIS,RECTUM_LABEL,INCLUDE_EMPTY_RECTUM,PRED_THRESHOLD,CENTER_COMPONENT_CONNECTIVITY)).encode()).hexdigest()
        cache_dir = OUTPUT_DIR / 'centre_cache'; cache_dir.mkdir(exist_ok=True)
        ids = sorted(patients)
        # Bounded chunks avoid accumulating all decompressed volumes in RAM.
        with ThreadPoolExecutor(max_workers=CPU_WORKERS) as pool:
            for start in tqdm(range(0,len(ids),CPU_WORKERS),desc='Patient cache / preprocessing'):
                chunk = ids[start:start+CPU_WORKERS]
                paths = {pid:cache_dir/(fingerprint(pid)+'.pt') for pid in chunk}
                futures = {pid:pool.submit(load_patient,pid) for pid in chunk if not paths[pid].exists()}
                for pid in chunk:
                    path = paths[pid]
                    if pid in futures:
                        d = assemble_patient(pid,futures[pid].result())
                        if d is not None:
                            # Store only tensors and basic Python types for safe loading.
                            tmp = path.with_suffix(".tmp")
                            torch.save((*d[:3],torch.from_numpy(d[3]),d[4]),tmp)
                            tmp.replace(path)
                    else:
                        d = torch.load(path,map_location='cpu',weights_only=True)
                        d = (*d[:3],d[3].numpy(),d[4])
                    if d is not None: data.append(d)
        if not data: raise ValueError('No eligible slices')
        self.images,self.masks,self.centres = [torch.cat(items) for items in zip(*(d[:3] for d in data))]
        self.kernel_probabilities = np.concatenate([d[3] for d in data])
        self.records = [r for d in data for r in d[4]]
    def __len__(self): return len(self.images)
    def __getitem__(self,i):
        return {'image':self.images[i],'mask':self.masks[i],'centre':self.centres[i],'index':i}

train_ds = RectumDataset(train_patients)
val_ds = RectumDataset(val_patients)
class TensorBatchLoader:
    """Vectorized batch gathering; no per-sample DataLoader overhead."""
    def __init__(self, ds, shuffle, device):
        self.ds, self.shuffle = ds, shuffle
        self.tensors = tuple(t.to(device) for t in (ds.images, ds.masks, ds.centres))
    def __iter__(self):
        device = self.tensors[0].device
        order = torch.randperm(len(self.ds),device=device) if self.shuffle else torch.arange(len(self.ds),device=device)
        for ids in order.split(BATCH_SIZE):
            x,y,c = (t.index_select(0,ids) for t in self.tensors)
            if device.type == "cpu" and DEVICE.type == "cuda":
                x,y,c = (t.pin_memory() for t in (x,y,c))
            yield dict(image=x,mask=y,centre=c,index=ids)
    def __len__(self): return (len(self.ds)+BATCH_SIZE-1)//BATCH_SIZE

storage = torch.device('cpu')
if DEVICE.type == 'cuda' and GPU_DATASET:
    required = sum(t.numel()*t.element_size() for ds in (train_ds,val_ds) for t in (ds.images,ds.masks,ds.centres))
    free,_ = torch.cuda.mem_get_info()
    if required < free * .35: storage = DEVICE
print('Training tensor storage:',storage)
train_loader = TensorBatchLoader(train_ds,True,storage)
val_loader = TensorBatchLoader(val_ds,False,storage)
for name,ds in [('train',train_ds),('val',val_ds)]:
    defined = [r for r in ds.records if r['gt_kernel_defined']]
    print(name, 'slices:',len(ds),'fallback rate:',np.mean([r['fallback'] for r in ds.records]),
          'centre in GT kernel (defined only):',np.mean([r['centre_in_kernel'] for r in defined]) if defined else float('nan'))
    (OUTPUT_DIR/f'{name}_centre_audit.json').write_text(json.dumps(ds.records,indent=2))


In [ ]:
fig,axes=plt.subplots(2,4,figsize=(14,7))
for ax,i in zip(axes.flat,np.linspace(0,len(val_ds)-1,min(8,len(val_ds)),dtype=int)):
    d=val_ds[i]; r,c=d['centre']; ax.imshow(d['image'][0],cmap='gray')
    if d['mask'].any(): ax.contour(d['mask'][0],levels=[0.5],colors='red')
    p=val_ds.kernel_probabilities[i].astype(np.float32)
    if p.min()<PRED_THRESHOLD<p.max(): ax.contour(p,levels=[PRED_THRESHOLD],colors='cyan')
    ax.scatter(c,r,c='orange',marker='x'); ax.set_title(f"{val_ds.records[i]['patient']} / {val_ds.records[i]['slice']}"); ax.axis('off')
plt.tight_layout()


## 5. Original trainable U-Net + star LBP layer
U-Net architecture based on `unet_star_segmentation.ipynb`; the star solver now uses the configurable DeepFlow LBP execution controls below. Float64 training; no additional sigmoid on LBP probabilities. The centre is fixed metadata. The source layer applies axis-wise directional constraints, not a Euclidean ray prior. No geometric augmentation by default, to preserve centre alignment.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class SimpleUNet(nn.Module):
    def __init__(self, in_channels=1, base=8):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, base)
        self.enc2 = DoubleConv(base, 2 * base)
        self.bottom = DoubleConv(2 * base, 4 * base)
        self.dec2 = DoubleConv(6 * base, 2 * base)
        self.dec1 = DoubleConv(3 * base, base)
        self.head = nn.Conv2d(base, 1, 1)
        # Begin near neutral unary scores to avoid saturating LBP at initialization.
        nn.init.normal_(self.head.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(F.max_pool2d(s1, 2))
        x = self.bottom(F.max_pool2d(s2, 2))
        x = F.interpolate(x, size=s2.shape[-2:], mode="bilinear", align_corners=False)
        x = self.dec2(torch.cat([x, s2], dim=1))
        x = F.interpolate(x, size=s1.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(self.dec1(torch.cat([x, s1], dim=1)))


class StarFinalLayer(nn.Module):
    def __init__(self, pairwise_weight=0.15):
        super().__init__()
        if pairwise_weight < 0:
            raise ValueError("Pairwise weights must be nonnegative.")
        self.pairwise_weight = float(pairwise_weight)
        # None is essential: model.eval() must still return soft marginals.
        self.solver = BinaryDeepFlow2dStar(
            test_type=None, check_every=STAR_CHECK_EVERY,
            compile_updates=STAR_COMPILE_UPDATES,
            record_history=STAR_RECORD_HISTORY,
        )

    def solver_inputs(self, scores, centres):
        if scores.ndim != 4 or scores.shape[1] != 1 or scores.dtype not in (torch.float32, torch.float64):
            raise ValueError("Expected float32 or float64 unary scores [B,1,H,W].")
        b, _, h, w = scores.shape
        centres = centres.to(device=scores.device, dtype=scores.dtype)
        if centres.shape != (b, 2):
            raise ValueError("Expected finite centres [B,2] in (row, column) order.")
        rows = torch.arange(h, device=scores.device, dtype=scores.dtype)[None, None, :, None]
        cols = torch.arange(w, device=scores.device, dtype=scores.dtype)[None, None, None, :]
        cy = centres[:, 0].view(b, 1, 1, 1)
        cx = centres[:, 1].view(b, 1, 1, 1)
        lx, ly = torch.zeros_like(scores), torch.zeros_like(scores)
        lx[:, :, :-1, :] = torch.sign(rows[:, :, :-1, :] + 0.5 - cy)
        ly[:, :, :, :-1] = torch.sign(cols[:, :, :, :-1] + 0.5 - cx)
        rx = torch.full_like(scores, self.pairwise_weight)
        ry = torch.full_like(scores, self.pairwise_weight)
        rx[:, :, -1, :] = 0
        ry[:, :, :, -1] = 0
        return tuple(t.contiguous() for t in (scores, rx, ry, lx, ly))

    def forward(self, scores, centres):
        probabilities, joint_rows, joint_cols = self.solver(*self.solver_inputs(scores, centres))
        return probabilities  # Joint marginals are not needed by Dice loss.


class UNetWithStar(nn.Module):
    def __init__(self, in_channels=1, base=8, pairwise_weight=0.15, score_scale=4.0):
        super().__init__()
        self.unet = SimpleUNet(in_channels, base)
        self.star = StarFinalLayer(pairwise_weight)
        self.score_scale = score_scale

    def unary_scores(self, images):
        with torch.autocast(device_type=images.device.type, dtype=torch.bfloat16, enabled=USE_AMP_UNET and TRAIN_DTYPE == torch.float32 and images.is_cuda and torch.cuda.is_bf16_supported()):
            logits = self.unet(images)
        return (self.score_scale * torch.tanh(logits.to(dtype=TRAIN_DTYPE))).contiguous()

    def forward(self, images, centres):
        return self.star(self.unary_scores(images), centres)


model = UNetWithStar(
    base=BASE_CHANNELS, pairwise_weight=PAIRWISE_WEIGHT, score_scale=SCORE_SCALE,
).to(device=DEVICE, dtype=TRAIN_DTYPE)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")
assert all(parameter.dtype == TRAIN_DTYPE for parameter in model.parameters())


### Training and solver controls

The existing float64 run settings, data paths, split and one-epoch configuration
remain unchanged. The fresh rectum U-Net is converted before Adam is constructed;
the frozen kernel predictor and cached tensors remain float32. There is no rectum
checkpoint restore in training and no global default-dtype change.

`STAR_CHECK_EVERY=100` retains checks at iteration 1, multiples of the interval and
the original cap. `STAR_COMPILE_UPDATES=True` opts into compiled tensor blocks;
first use includes compilation. Successful solver/library calls are silent.

`STAR_RECORD_HISTORY=False` keeps final scalar summaries in `last_convergence`
without accumulating history during training. The smoke test temporarily requests
histories and restores the normal options even if a check fails. Each invocation
owns its forward/backward trace. Residual plots use the saved smoke trace, not the
latest training call.

The original message/adjoint equations, damping, epsilon values, sum-based forward
criterion, relative backward criterion and iteration-budget rules are preserved.
`iteration_limit` means the requested residual criterion was not met. No dtype is
forced inside the solver: buffers inherit the input dtype.

The two research experiments below are optional and disabled by default: a tiny
eager/compiled comparison and a frozen-input float32/float64 checkpoint comparison.
They are separate from ordinary training and report their results explicitly.
The native hard MAP inference path remains unchanged.


### Optional eager/compiled preflight

Enable only when checking compilation on this environment. It compares a tiny fixed
problem and unary/pairwise gradients; agreement does not prove pairwise-gradient
correctness. No training step is taken and normal histories remain disabled.


In [ ]:
RUN_COMPILED_PREFLIGHT = False
if RUN_COMPILED_PREFLIGHT:
    sample_scores = torch.linspace(-0.6, 0.9, 20, device=DEVICE, dtype=TRAIN_DTYPE).reshape(1, 1, 4, 5)
    sample_centres = torch.tensor([[1., 2.]], device=DEVICE, dtype=TRAIN_DTYPE)
    preflight_results = []
    for compiled in (False, True):
        d = sample_scores.clone().requires_grad_()
        d, rx, ry, lx, ly = model.star.solver_inputs(d, sample_centres)
        rx.requires_grad_(); ry.requires_grad_()
        candidate = BinaryDeepFlow2dStar(test_type=None, check_every=STAR_CHECK_EVERY,
                                        compile_updates=compiled, record_history=False)
        p = candidate(d, rx, ry, lx, ly)[0]
        gradients = torch.autograd.grad(p.square().mean(), (d, rx, ry))
        preflight_results.append((p.detach(), *gradients))
    rtol, atol = (1e-7, 1e-9) if TRAIN_DTYPE == torch.float64 else (1e-4, 1e-6)
    differences = {}
    for name, eager, compiled in zip(("probabilities", "unary_gradient", "row_gradient", "column_gradient"), *preflight_results):
        torch.testing.assert_close(compiled, eager, rtol=rtol, atol=atol)
        differences[name] = (eager-compiled).abs().max().item()
    print("Compiled preflight max absolute differences:", differences)


In [ ]:
def dice_per_sample(probabilities, targets, smooth=1e-6):
    if probabilities.shape != targets.shape:
        raise ValueError("Prediction and target shapes must match.")
    dims = (1, 2, 3)
    intersection = (probabilities * targets).sum(dim=dims)
    return (2 * intersection + smooth) / (
        probabilities.sum(dim=dims) + targets.sum(dim=dims) + smooth
    )


class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, probabilities, targets):
        return 1 - dice_per_sample(probabilities, targets, self.smooth).mean()


criterion = DiceLoss(smooth=1e-6)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


def move_batch(batch):
    return (
        batch["image"].to(DEVICE, dtype=TRAIN_DTYPE, non_blocking=True),
        batch["mask"].to(DEVICE, dtype=TRAIN_DTYPE, non_blocking=True),
        batch["centre"].to(DEVICE, dtype=TRAIN_DTYPE, non_blocking=True),
    )


def check_probabilities(probabilities):
    if not torch.isfinite(probabilities).all():
        raise FloatingPointError("DeepFlow returned non-finite probabilities.")
    if ((probabilities < 0) | (probabilities > 1)).any():
        raise FloatingPointError("Expected probabilities in [0, 1].")


## 6. One-batch dtype, gradient and timing check

One forward/backward checks the configured dtype and nonzero finite gradients,
without an optimizer step. CUDA is synchronized at stage boundaries. The single
stage table includes Python dispatch and first-use compilation; repeat for warmed
timing. Dice/star backward is isolated from U-Net backward at the unary scores.
Histories are explicitly enabled only for this diagnostic call and then restored.


In [ ]:
from dataclasses import replace
from time import perf_counter

model.train()
optimizer.zero_grad(set_to_none=True)
images, targets, centres = move_batch(next(iter(train_loader)))
star_timing_seconds = {}


def measure_star_stage(name, operation):
    if images.is_cuda:
        torch.cuda.synchronize(images.device)
    started = perf_counter()
    result = operation()
    if images.is_cuda:
        torch.cuda.synchronize(images.device)
    star_timing_seconds[name] = perf_counter() - started
    return result


normal_options = model.star.solver.lbp_options
model.star.solver.lbp_options = replace(normal_options, record_history=True)
try:
    scores = measure_star_stage("unet_forward", lambda: model.unary_scores(images))
    assert images.dtype == targets.dtype == centres.dtype == scores.dtype == TRAIN_DTYPE
    probabilities = measure_star_stage("star_forward", lambda: model.star(scores, centres))
    check_probabilities(probabilities)
    assert probabilities.shape == targets.shape and probabilities.requires_grad
    assert probabilities.dtype == TRAIN_DTYPE
    loss = measure_star_stage("dice_forward", lambda: criterion(probabilities, targets))

    # Stop at scores to isolate Dice + star backward from U-Net backward.
    score_gradient = measure_star_stage(
        "star_backward_with_dice", lambda: torch.autograd.grad(loss, scores)[0],
    )
    assert torch.isfinite(score_gradient).all() and score_gradient.abs().sum().item() > 0
    scores.retain_grad()
    measure_star_stage("unet_backward", lambda: scores.backward(score_gradient))
    star_smoke_convergence = copy.deepcopy(model.star.solver.last_convergence)
    for gradient in (scores.grad, model.unet.enc1.net[0].weight.grad):
        assert gradient is not None and gradient.dtype == TRAIN_DTYPE
        assert torch.isfinite(gradient).all() and gradient.abs().sum().item() > 0
finally:
    model.star.solver.lbp_options = normal_options
    optimizer.zero_grad(set_to_none=True)

print(f"Smoke timing: {images.device}, {TRAIN_DTYPE}, batch={tuple(images.shape)}, "
      f"check_every={normal_options.check_every}, compiled={normal_options.compile_updates}")
measured_training_seconds = sum(star_timing_seconds.values())
for name, seconds in star_timing_seconds.items():
    share = 100 * seconds / measured_training_seconds if measured_training_seconds else 0.0
    print(f"  {name:28s} {seconds:10.3f} s  ({share:5.1f}%)")
print(f"  {'sum of measured stages':28s} {measured_training_seconds:10.3f} s")


### Inspect the smoke residuals

These plots use the smoke test's saved scalar histories. Normal training retains
summaries only. A decreasing residual shows progress; hitting the cap is not a
claim of convergence. The JSON contains residuals, dtype fields and terminal summaries.


In [ ]:
# Use the smoke snapshot: later training calls do not accumulate these histories.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for phase, ax in zip(("forward", "backward"), axes):
    trace = star_smoke_convergence[phase]
    history = trace.get("history", [])
    if history:
        ax.semilogy([row["iteration"] for row in history],
                    np.maximum([row["residual"] for row in history], 1e-30), label="Residual")
        ax.axhline(max(trace["tolerance"], 1e-30), color="red", linestyle="--", label="Tolerance")
        ax.legend()
    else:
        ax.text(.05, .5, "No history was requested for this phase", transform=ax.transAxes)
    ax.set_title(f"{phase.capitalize()}: {trace['status']}")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Residual")
    ax.grid(alpha=.2)
    print(f"{phase}: {trace['status']}, iterations={trace['iterations']}/{trace['iteration_limit']}, "
          f"residual={trace['residual']:.6g}, tolerance={trace['tolerance']:.3g}")
plt.tight_layout()
plt.show()
convergence_path = OUTPUT_DIR / "star_smoke_convergence.json"
convergence_path.write_text(json.dumps(star_smoke_convergence, indent=2, allow_nan=False))


## 7. Train only the rectum model

In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    totals = torch.zeros(3, device=DEVICE, dtype=TRAIN_DTYPE)
    count = 0
    with torch.set_grad_enabled(training):
        for step,batch in enumerate(tqdm(loader,leave=False,desc='Train' if training else 'Validation')):
            images, targets, centres = move_batch(batch)
            if training: optimizer.zero_grad(set_to_none=True)
            probabilities = model(images, centres)
            soft = dice_per_sample(probabilities, targets)
            loss = 1 - soft.mean()
            if step % CHECK_EVERY == 0:
                check_probabilities(probabilities)
                if not torch.isfinite(loss): raise FloatingPointError('Non-finite loss')
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(),5.,error_if_nonfinite=True)
                optimizer.step()
            with torch.no_grad():
                hard = dice_per_sample((probabilities >= .5).to(targets.dtype),targets).sum() if not training else torch.zeros((),device=DEVICE,dtype=TRAIN_DTYPE)
                totals += torch.stack((loss.detach()*len(images),soft.detach().sum(),hard))
            count += len(images)
    if not count: raise ValueError('Empty loader')
    values = (totals/count).cpu().tolist()
    if not np.isfinite(values).all(): raise FloatingPointError('Non-finite epoch metrics')
    return dict(loss=values[0],soft_dice=values[1],hard_dice=values[2])

def train_one_epoch(model,loader,optimizer,criterion):
    return run_epoch(model,loader,optimizer)

def validate(model,loader,criterion):
    return run_epoch(model,loader)

history = []
best_val_loss = float("inf")
best_state = None
for epoch in range(1, EPOCHS + 1):
    train_metrics = train_one_epoch(model, train_loader, optimizer, criterion)
    val_metrics = validate(model, val_loader, criterion)
    history.append({"epoch": epoch, "train": train_metrics, "val": val_metrics})
    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        torch.save({
            "epoch": epoch,
            "model_state_dict": best_state,
            "optimizer_state_dict": copy.deepcopy(optimizer.state_dict()),
            "val_loss": best_val_loss,
            "config": {
                "in_channels": 1, "base": BASE_CHANNELS,
                "training_dtype": str(TRAIN_DTYPE),
                "pairwise_weight": PAIRWISE_WEIGHT, "score_scale": SCORE_SCALE,
                "image_size": IMAGE_SIZE, "rotate_k": ROTATE_K, "seed": SEED,
                "star_check_every": STAR_CHECK_EVERY,
                "star_compile_updates": STAR_COMPILE_UPDATES,
                "star_record_history": STAR_RECORD_HISTORY,
                "kernel_checkpoint": str(KERNEL_CHECKPOINT), "kernel_threshold": PRED_THRESHOLD,
                "train_patients": sorted(train_patients), "val_patients": sorted(val_patients),
            },
        }, CHECKPOINT_PATH)
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | train loss {train_metrics['loss']:.4f} | "
        f"val loss {val_metrics['loss']:.4f} | "
        f"val soft Dice {val_metrics['soft_dice']:.4f} | "
        f"val binary Dice {val_metrics['hard_dice']:.4f}"
    )

if best_state is not None:
    model.load_state_dict(best_state)
model.eval()
print("Best checkpoint:", CHECKPOINT_PATH.resolve())


### Optional frozen-input float32/float64 reference

Run separately after training (or with a saved checkpoint). This diagnostic is disabled
by default and takes no optimizer step. It loads a checkpoint into a separate copy,
freezes a common float32 representation of the scores/weights/directions, and compares
float32/float64 solvers. It retains histories explicitly through the research helper.
The active model and optimizer are untouched. Default eager execution isolates dtype
from compilation rounding; timings include first-use compilation if enabled.

Inspect `float64_converged` before treating the higher-precision result as a converged
reference. This test is not an exact-posterior guarantee. JSON, CPU tensor artifacts
and plots preserve the comparison. A missing checkpoint raises an error instead of
silently using an untrained network.


In [ ]:
# Float64 reference: no optimizer step and no changes to the active training model.
from binary_lbp_reference import compare_binary_star_precision
import hashlib

RUN_FLOAT64_REFERENCE = False
FLOAT64_REFERENCE_USE_CHECKPOINT = True
FLOAT64_REFERENCE_IMAGES = 1  # Preserve H,W and geometry; limit batch size only.
FLOAT64_REFERENCE_COMPILE = False  # Eager in BOTH runs isolates the dtype comparison.

if RUN_FLOAT64_REFERENCE:
    if FLOAT64_REFERENCE_IMAGES < 1:
        raise ValueError("FLOAT64_REFERENCE_IMAGES must be positive")
    reference_model = copy.deepcopy(model).eval()
    reference_metadata = {"weight_source": "current_model"}
    if FLOAT64_REFERENCE_USE_CHECKPOINT:
        if not CHECKPOINT_PATH.is_file():
            raise FileNotFoundError(f"Trained checkpoint not found: {CHECKPOINT_PATH}. Set CHECKPOINT_PATH explicitly.")
        reference_checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
        saved_state = reference_checkpoint["model_state_dict"]
        checkpoint_dtype = next(value.dtype for value in saved_state.values() if value.is_floating_point())
        reference_model.to(dtype=checkpoint_dtype)
        reference_model.load_state_dict(saved_state, strict=True)
        checkpoint_config = reference_checkpoint.get("config", {})
        reference_model.star.pairwise_weight = float(checkpoint_config.get("pairwise_weight", reference_model.star.pairwise_weight))
        reference_model.score_scale = float(checkpoint_config.get("score_scale", reference_model.score_scale))
        reference_metadata = dict(
            weight_source="checkpoint", checkpoint=str(CHECKPOINT_PATH.resolve()),
            checkpoint_sha256=hashlib.sha256(CHECKPOINT_PATH.read_bytes()).hexdigest(),
            checkpoint_epoch=reference_checkpoint.get("epoch"),
        )
        print("Float64 reference checkpoint:", reference_metadata["checkpoint"],
              "epoch:", reference_metadata["checkpoint_epoch"])
    else:
        print("Float64 reference uses current weights; ensure this model has been trained.")
    reference_batch = next(iter(val_loader))
    reference_images, reference_targets, reference_centres = move_batch(reference_batch)
    reference_images = reference_images[:FLOAT64_REFERENCE_IMAGES]
    reference_targets = reference_targets[:FLOAT64_REFERENCE_IMAGES]
    reference_centres = reference_centres[:FLOAT64_REFERENCE_IMAGES]
    with torch.no_grad():
        model_dtype = next(reference_model.parameters()).dtype
        reference_logits = reference_model.unet(reference_images.to(dtype=model_dtype))
        reference_scores = (reference_model.score_scale * torch.tanh(reference_logits)).float()
        reference_inputs = reference_model.star.solver_inputs(reference_scores, reference_centres)
    reference_metadata.update(
        image_shape=list(reference_images.shape), centres_rc=reference_centres.cpu().tolist(),
        score_scale=reference_model.score_scale, pairwise_weight=reference_model.star.pairwise_weight,
    )
    if "index" in reference_batch:
        reference_metadata["validation_indices"] = reference_batch["index"][:len(reference_images)].cpu().tolist()
    float_precision_report, float_precision_artifacts = compare_binary_star_precision(
        reference_inputs, reference_targets, check_every=STAR_CHECK_EVERY,
        compile_updates=FLOAT64_REFERENCE_COMPILE,
        smooth=criterion.smooth, metadata=reference_metadata,
    )
    float_precision_artifacts["metadata"] = reference_metadata
    float_precision_artifacts["centres_rc"] = reference_centres.detach().cpu()
    precision_json_path = OUTPUT_DIR / "star_float32_float64_comparison.json"
    precision_tensor_path = OUTPUT_DIR / "star_float32_float64_inputs_outputs.pt"
    precision_json_path.write_text(json.dumps(float_precision_report, indent=2, allow_nan=False))
    torch.save(float_precision_artifacts, precision_tensor_path)
    print("Differences:", float_precision_report["differences"])
    print("Float64 met both convergence criteria:", float_precision_report["float64_converged"])
    print("Saved report:", precision_json_path.resolve())
    print("Saved frozen inputs and outputs:", precision_tensor_path.resolve())
    del reference_model  # The original trained model and optimizer remain intact.


In [ ]:
# Inspect spatial errors and compare residual histories after the reference cell.
if RUN_FLOAT64_REFERENCE:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    artifact32, artifact64 = float_precision_artifacts["float32"], float_precision_artifacts["float64"]
    p32, p64 = artifact32["probabilities"][0, 0].double(), artifact64["probabilities"][0, 0]
    for ax, panel, title in zip(axes[0], (p32, p64, (p32-p64).abs()),
                                ("Float32 probability", "Float64 probability", "Absolute probability difference")):
        kwargs = {"vmin": 0, "vmax": 1} if "difference" not in title else {}
        im = ax.imshow(panel.numpy(), cmap="viridis", **kwargs)
        ax.set_title(title)
        ax.axis("off")
        fig.colorbar(im, ax=ax, shrink=.7)
    for ax, phase in zip(axes[1, :2], ("forward", "backward")):
        for name in ("float32", "float64"):
            trace = float_precision_report[name]["trace"][phase]
            history = trace.get("history", [])
            if history:
                ax.semilogy([row["iteration"] for row in history],
                            np.maximum([row["residual"] for row in history], 1e-30),
                            label=f"{name}: {trace['status']}")
            ax.axhline(max(trace["tolerance"], 1e-30), linestyle="--", alpha=.5,
                       label=f"{name} tolerance")
        ax.set_title(f"{phase.capitalize()} residuals")
        ax.set_xlabel("Iteration")
        ax.legend(fontsize=8)
    g32 = artifact32["unary_gradient"][0, 0].double()
    g64 = artifact64["unary_gradient"][0, 0]
    im = axes[1, 2].imshow((g32-g64).abs().numpy(), cmap="magma")
    axes[1, 2].set_title("Absolute unary-gradient difference")
    axes[1, 2].axis("off")
    fig.colorbar(im, ax=axes[1, 2], shrink=.7)
    plt.tight_layout()
    precision_figure_path = OUTPUT_DIR / "star_float32_float64_comparison.png"
    fig.savefig(precision_figure_path, dpi=150)
    plt.show()
    print("Saved comparison figure:", precision_figure_path.resolve())


## 8. Validation diagnostics
Report thresholded LBP Dice by patient and by centre validity, with the unconstrained unary scores as a diagnostic (not a separately trained baseline).

In [ ]:
import csv
rows=[]
model.eval()
with torch.no_grad():
    for batch in tqdm(val_loader,desc='Validation'):
        images,targets,centres=move_batch(batch)
        scores=model.unary_scores(images)
        p=model.star(scores,centres)
        check_probabilities(p)
        hard=(p>=0.5).to(targets.dtype)
        dice=dice_per_sample(hard,targets).cpu().numpy()
        unary=dice_per_sample((scores>=0).to(targets.dtype),targets).cpu().numpy()
        inter=(hard*targets).sum((1,2,3)); union=(hard+targets-hard*targets).sum((1,2,3))
        iou=((inter+1e-6)/(union+1e-6)).cpu().numpy()
        for j,i in enumerate(batch['index'].tolist()):
            rows.append(dict(val_ds.records[i],dice=float(dice[j]),iou=float(iou[j]),unary_dice=float(unary[j])))
with open(OUTPUT_DIR/'validation.csv','w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
print('Mean slice Dice:',np.mean([r['dice'] for r in rows]))
print('Mean slice IoU:',np.mean([r['iou'] for r in rows]))
patient_dice={pid:np.mean([r['dice'] for r in rows if r['patient']==pid]) for pid in sorted({r['patient'] for r in rows})}
print('Patient-macro slice Dice:',np.mean(list(patient_dice.values())))
for name,group in [('valid centre',[r for r in rows if r['gt_kernel_defined'] and r['centre_in_kernel']]),('invalid centre',[r for r in rows if r['gt_kernel_defined'] and not r['centre_in_kernel']]),('fallback',[r for r in rows if r['fallback']])]:
    print(name,len(group),'Dice:',np.mean([r['dice'] for r in group]) if group else float('nan'))
plt.figure(figsize=(8,3));plt.plot([h['train']['loss'] for h in history],label='train');plt.plot([h['val']['loss'] for h in history],label='val');plt.legend();plt.ylabel('Dice loss');plt.xlabel('Epoch')


In [ ]:
fig,axes=plt.subplots(4,5,figsize=(15,11))
for axrow,i in zip(axes,np.linspace(0,len(val_ds)-1,min(4,len(val_ds)),dtype=int)):
    d=val_ds[i]; x=d['image'][None].to(DEVICE,dtype=TRAIN_DTYPE); c=d['centre'][None].to(DEVICE)
    with torch.no_grad():
        scores=model.unary_scores(x); p=model.star(scores,c)[0,0].cpu().numpy()
    for ax,arr,title in zip(axrow,[d['image'][0],val_ds.kernel_probabilities[i],d['mask'][0],scores[0,0].cpu(),p>=0.5],['MRI','Predicted kernel','GT rectum','Unary scores','Star LBP mask']):
        ax.imshow(arr,cmap='gray');ax.set_title(title);ax.axis('off')
    axrow[1].scatter(d['centre'][1],d['centre'][0],c='orange',marker='x')
plt.tight_layout()


## 9. End-to-end inference on an unseen MRI slice
Input is a raw 2D slice in the same orientation as training. Output is on the 128×128 model grid; no annotation or kernel label is required. For full-volume inference call on every slice, and restore geometry separately. Optional hard MAP uses the original CPU extension and is not used for training.
Input must be in the original canonical slice orientation; this function rotates it once. Outputs and centres are in the rotated model grid. Use `np.rot90(mask, -ROTATE_K)` for the original resized orientation.

In [ ]:
@torch.no_grad()
def segment_rectum(raw_slice, hard_map=False):
    model.eval()
    x=prep.normalize_slice(resize_2d(np.rot90(raw_slice, ROTATE_K)))
    x=torch.from_numpy(np.asarray(x,dtype=np.float32)[None,None])
    kp,c,f=predict_centres(x)
    centres=torch.from_numpy(c).to(DEVICE)
    scores=model.unary_scores(x.to(DEVICE,dtype=TRAIN_DTYPE))
    probabilities=model.star(scores,centres)
    check_probabilities(probabilities)
    result={'kernel_probability':kp[0], 'centre_rc':c[0], 'fallback':bool(f[0]),
            'rectum_probability':probabilities[0,0].cpu().numpy(),
            'rectum_mask':(probabilities[0,0]>=0.5).cpu().numpy()}
    if hard_map:
        if not hasattr(deepflow,'binary_auglag_2dstar_cpu_forward'): raise RuntimeError('Missing CPU MAP binding')
        solver=BinaryDeepFlow2dStar(test_type='MAP').eval()
        result['map_mask']=solver(*model.star.solver_inputs(scores.cpu().float(),centres.cpu().float()))[0,0].numpy()
    return result
